In [ ]:
import subprocess, sys, os

def pip_install(*pkgs):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs])
    if result.returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                         "--break-system-packages", *pkgs], check=True)


pip_install("faiss-cpu", "sentence-transformers")

import numpy as np
import pandas as pd
import time
import re


In [ ]:
CANDIDATE_PATHS = [
    "/kaggle/input/datasets/mrnotalent/laptop-embedding/laptop_chunks_embeddings_with_lineage.parquet",
    "laptop_chunks_embeddings_with_lineage.parquet",
    "/content/laptop_chunks_embeddings_with_lineage.parquet",       
    "/mnt/user-data/uploads/laptop_chunks_embeddings_with_lineage.parquet",
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Couldn't find laptop_chunks_embeddings_with_lineage.parquet in any of the expected "
        f"locations: {CANDIDATE_PATHS}. Upload it or add its real path to CANDIDATE_PATHS."
    )

df = pd.read_parquet(DATA_PATH)
print("loaded from:", DATA_PATH)
print(df.shape)

embeddings = np.stack(df["embedding"].to_numpy()).astype("float32")
print("embedding matrix:", embeddings.shape, embeddings.dtype)

device_level = df.drop_duplicates(subset="row_uid").copy()
print("unique devices:", len(device_level))
print(f"avg chunks/device: {len(df) / len(device_level):.2f} (max {df.groupby('row_uid').size().max()})")


loaded from: laptop_chunks_embeddings_with_lineage.parquet
(4171, 22)
embedding matrix: (4171, 384) float32
unique devices: 1960
avg chunks/device: 2.13 (max 9)


In [3]:
oled_row_uids = set(df.loc[df["chunk_text"].str.contains("OLED", case=False, na=False), "row_uid"])

hand_queries_v2 = [
    {"query": "cheap laptop under $400 for basic tasks",
     "filter": lambda d: d["price_usd"] < 400},
    {"query": "gaming laptop with NVIDIA RTX under $1200",
     "filter": lambda d: d["gpu"].str.contains("RTX", case=False, na=False) & (d["price_usd"] < 1200)},
    {"query": "lightweight ultrabook under 3 lbs",
     "filter": lambda d: d["category"].str.contains("Ultrabook|Thin", case=False, na=False)},
    {"query": "laptop with 32GB RAM and 1TB SSD",
     "filter": lambda d: (d["ram_gb"] >= 32) & d["storage"].str.contains("1TB|1 TB", case=False, na=False)},
    {"query": "AMD Ryzen laptop under $600",
     "filter": lambda d: d["cpu"].str.contains("Ryzen", case=False, na=False) & (d["price_usd"] < 600)},
    {"query": "Intel Core i7 laptop with NVIDIA graphics",
     "filter": lambda d: d["cpu"].str.contains("i7", case=False, na=False) & d["gpu"].str.contains("NVIDIA|RTX|GeForce", case=False, na=False)},
    {"query": "Intel Core i5 laptop under $700",
     "filter": lambda d: d["cpu"].str.contains("i5", case=False, na=False) & (d["price_usd"] < 700)},
    {"query": "laptop with a 1TB SSD under $900",
     "filter": lambda d: d["storage"].str.contains("1TB|1 TB", case=False, na=False) & d["storage"].str.contains("SSD|PCIe|NVMe", case=False, na=False) & (d["price_usd"] < 900)},
    {"query": "touchscreen 2-in-1 convertible laptop",
     "filter": lambda d: d["display"].str.contains("Touch", case=False, na=False) & d["category"].str.contains("2-in-1|Convertible|Yoga", case=False, na=False)},
    {"query": "premium laptop over $1500 with 32GB RAM",
     "filter": lambda d: (d["price_usd"] > 1500) & (d["ram_gb"] >= 32)},
    {"query": "Dell business laptop under $1000",
     "filter": lambda d: d["title"].str.contains("Dell", case=False, na=False) & (d["price_usd"] < 1000)},
    {"query": "HP laptop with AMD Ryzen processor",
     "filter": lambda d: d["title"].str.contains("HP", case=False, na=False) & d["cpu"].str.contains("Ryzen", case=False, na=False)},
    {"query": "Lenovo laptop under $500",
     "filter": lambda d: d["title"].str.contains("Lenovo", case=False, na=False) & (d["price_usd"] < 500)},
    {"query": "ASUS laptop with OLED display",
     "filter": lambda d, _oled=oled_row_uids: d["title"].str.contains("ASUS", case=False, na=False) & d["row_uid"].isin(_oled)},
    {"query": "Acer laptop under $500",
     "filter": lambda d: d["title"].str.contains("Acer", case=False, na=False) & (d["price_usd"] < 500)},
    {"query": "17 inch gaming laptop",
     "filter": lambda d: d["display"].str.contains("17", na=False) & d["gpu"].str.contains("NVIDIA|RTX|GeForce", case=False, na=False)},
    {"query": "compact 13 inch laptop under $800",
     "filter": lambda d: d["display"].str.contains("13", na=False) & (d["price_usd"] < 800)},
    {"query": "15.6 inch Full HD budget laptop under $500",
     "filter": lambda d: d["display"].str.contains("15.6", na=False) & d["display"].str.contains("FHD|Full HD", case=False, na=False) & (d["price_usd"] < 500)},
    {"query": "workstation laptop with professional GPU",
     "filter": lambda d: d["gpu"].str.contains("Quadro|RTX A", case=False, na=False)},
    {"query": "laptop with integrated graphics only under $400",
     "filter": lambda d: d["gpu"].str.contains("Intel", case=False, na=False) & ~d["gpu"].str.contains("NVIDIA|AMD|Radeon", case=False, na=False) & (d["price_usd"] < 400)},
    {"query": "laptop with over 10 hours battery life",
     "filter": lambda d: d["battery"].notna() & d["battery"].str.contains(r"1[0-9]\s*Hour|[2-9][0-9]\s*Hour", case=False, na=False, regex=True)},
    {"query": "Chromebook under $300",
     "filter": lambda d: d["title"].str.contains("Chromebook", case=False, na=False)},
    {"query": "512GB SSD laptop under $600",
     "filter": lambda d: d["storage"].str.contains("512", na=False) & d["storage"].str.contains("SSD|PCIe|NVMe", case=False, na=False) & (d["price_usd"] < 600)},
    {"query": "Intel Core Ultra laptop with 16GB RAM",
     "filter": lambda d: d["cpu"].str.contains("Core Ultra", case=False, na=False) & (d["ram_gb"] >= 16)},
    {"query": "laptop under $300 for basic browsing",
     "filter": lambda d: d["price_usd"] < 300},
    {"query": "mid-range laptop $700-$1000 with SSD",
     "filter": lambda d: (d["price_usd"] >= 700) & (d["price_usd"] <= 1000) & d["storage"].str.contains("SSD|PCIe|NVMe", case=False, na=False)},
    {"query": "gaming laptop under $1000",
     "filter": lambda d: d["gpu"].str.contains("NVIDIA|RTX|GeForce", case=False, na=False) & (d["price_usd"] < 1000)},
    {"query": "business laptop 14 inch under $1200",
     "filter": lambda d: d["display"].str.contains("14", na=False) & (d["price_usd"] < 1200)},
    {"query": "budget student laptop 8GB RAM",
     "filter": lambda d: (d["ram_gb"] >= 8) & (d["ram_gb"] < 16) & (d["price_usd"] < 500)},
    {"query": "high performance laptop 64GB RAM",
     "filter": lambda d: d["ram_gb"] >= 64},
]

eval_rows = []
for item in hand_queries_v2:
    mask = item["filter"](device_level)
    relevant_uids = device_level.loc[mask, "row_uid"].tolist()
    eval_rows.append({
        "query": item["query"],
        "n_relevant": len(relevant_uids),
        "relevant_row_uids": relevant_uids,
    })

eval_df_correct = pd.DataFrame(eval_rows)
print(eval_df_correct[["query", "n_relevant"]].to_string())

zero_hit = eval_df_correct[eval_df_correct["n_relevant"] == 0]
if len(zero_hit):
    print(f"\nWARNING: queries with zero matching devices out of {len(hand_queries_v2)} — filter logic may need adjustment for this data:")
    print(zero_hit["query"].tolist())
else:
    print(f"\nAll {len(hand_queries_v2)} queries have at least one matching device — eval set is usable.")


                                              query  n_relevant
0           cheap laptop under $400 for basic tasks         242
1         gaming laptop with NVIDIA RTX under $1200           6
2                 lightweight ultrabook under 3 lbs         156
3                  laptop with 32GB RAM and 1TB SSD         194
4                       AMD Ryzen laptop under $600          39
5         Intel Core i7 laptop with NVIDIA graphics         224
6                   Intel Core i5 laptop under $700         162
7                  laptop with a 1TB SSD under $900          30
8             touchscreen 2-in-1 convertible laptop          97
9           premium laptop over $1500 with 32GB RAM         213
10                 Dell business laptop under $1000         240
11               HP laptop with AMD Ryzen processor          59
12                         Lenovo laptop under $500          73
13                    ASUS laptop with OLED display          12
14                           Acer laptop

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
query_embeddings = model.encode(eval_df_correct["query"].tolist(), batch_size=32, show_progress_bar=False).astype("float32")
print(query_embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(30, 384)


In [ ]:
import faiss

d = embeddings.shape[1] 
n = embeddings.shape[0]
build_stats = {}

def build_and_time(build_fn, name):
    t0 = time.perf_counter()
    index = build_fn()
    build_time_s = time.perf_counter() - t0
    mem_mb = len(faiss.serialize_index(index)) / (1024 ** 2)
    build_stats[name] = (build_time_s, mem_mb)
    print(f"{name:20s} | build: {build_time_s:.3f}s | memory: {mem_mb:.2f} MB")
    return index

def _build_flat():
    idx = faiss.IndexFlatL2(d)
    idx.add(embeddings)
    return idx
index_flat = build_and_time(_build_flat, "IndexFlatL2")

def _build_ivf():
    nlist_local = min(50, n // 10)
    quantizer = faiss.IndexFlatL2(d)
    idx = faiss.IndexIVFFlat(quantizer, d, nlist_local)
    idx.train(embeddings)
    idx.add(embeddings)
    idx.nprobe = 8
    return idx
index_ivf = build_and_time(_build_ivf, "IndexIVFFlat")

def _build_hnsw():
    idx = faiss.IndexHNSWFlat(d, 32)
    idx.hnsw.efConstruction = 40
    idx.add(embeddings)
    idx.hnsw.efSearch = 32
    return idx
index_hnsw = build_and_time(_build_hnsw, "IndexHNSWFlat")

m = 8
nbits = max(nb for nb in range(2, 9) if 39 * (2 ** nb) <= n)
print(f"using nbits={nbits} for PQ/IVFPQ ({2**nbits} centroids, needs >= {39*2**nbits} training pts, have {n})")

def _build_pq():
    idx = faiss.IndexPQ(d, m, nbits)
    idx.train(embeddings)
    idx.add(embeddings)
    return idx
index_pq = build_and_time(_build_pq, "IndexPQ")

nlist = min(64, max(8, n // 10))
def _build_ivfpq():
    quantizer_pq = faiss.IndexFlatL2(d)
    idx = faiss.IndexIVFPQ(quantizer_pq, d, nlist, m, nbits)
    idx.train(embeddings)
    idx.add(embeddings)
    idx.nprobe = 8
    return idx
index_ivfpq = build_and_time(_build_ivfpq, "IndexIVFPQ")

print("\nAll 5 indices built:", index_flat.ntotal, index_ivf.ntotal, index_hnsw.ntotal, index_pq.ntotal, index_ivfpq.ntotal)


IndexFlatL2          | build: 0.005s | memory: 6.11 MB
IndexIVFFlat         | build: 0.063s | memory: 6.22 MB
IndexHNSWFlat        | build: 0.350s | memory: 7.19 MB
using nbits=6 for PQ/IVFPQ (64 centroids, needs >= 2496 training pts, have 4171)
IndexPQ              | build: 0.212s | memory: 0.12 MB
IndexIVFPQ           | build: 0.307s | memory: 0.24 MB

All 5 indices built: 4171 4171 4171 4171 4171


In [6]:
def evaluate_index(index, name, query_vecs, k=5):
    _, gt_indices = index_flat.search(query_vecs, k)

    start = time.perf_counter()
    _, pred_indices = index.search(query_vecs, k)
    elapsed = time.perf_counter() - start
    latency_ms_per_query = (elapsed / len(query_vecs)) * 1000

    hits, total = 0, 0
    for gt_row, pred_row in zip(gt_indices, pred_indices):
        hits += len(set(gt_row.tolist()) & set(pred_row.tolist()))
        total += len(gt_row)
    recall_at_k = hits / total

    build_time_s, mem_mb = build_stats[name]
    print(f"{name:20s} | recall@{k}: {recall_at_k:.3f} | avg latency: {latency_ms_per_query:.3f} ms/query "
          f"| build: {build_time_s:.3f}s | memory: {mem_mb:.2f} MB")
    return {"index": name, "recall_at_5": recall_at_k, "latency_ms": latency_ms_per_query,
            "build_time_s": build_time_s, "memory_mb": mem_mb}

results = []
for idx, name in [(index_flat, "IndexFlatL2"), (index_ivf, "IndexIVFFlat"), (index_hnsw, "IndexHNSWFlat"),
                   (index_pq, "IndexPQ"), (index_ivfpq, "IndexIVFPQ")]:
    results.append(evaluate_index(idx, name, query_embeddings, k=5))

results_df = pd.DataFrame(results)
results_df.to_csv("m3_index_comparison_5way.csv", index=False)
results_df


IndexFlatL2          | recall@5: 1.000 | avg latency: 0.363 ms/query | build: 0.005s | memory: 6.11 MB
IndexIVFFlat         | recall@5: 0.827 | avg latency: 0.096 ms/query | build: 0.063s | memory: 6.22 MB
IndexHNSWFlat        | recall@5: 0.847 | avg latency: 0.116 ms/query | build: 0.350s | memory: 7.19 MB
IndexPQ              | recall@5: 0.133 | avg latency: 0.123 ms/query | build: 0.212s | memory: 0.12 MB
IndexIVFPQ           | recall@5: 0.220 | avg latency: 0.060 ms/query | build: 0.307s | memory: 0.24 MB


,index,recall_at_5,latency_ms,build_time_s,memory_mb
0,IndexFlatL2,1.000000,0.363015,0.005100,6.109906
1,IndexIVFFlat,0.826667,0.095990,0.063356,6.215442
2,IndexHNSWFlat,0.846667,0.115586,0.349854,7.189672
3,IndexPQ,0.133333,0.123185,0.211869,0.117699
4,IndexIVFPQ,0.220000,0.060093,0.306935,0.243849


In [7]:
_, I_diag = index_flat.search(query_embeddings, 5)
unique_device_counts = pd.Series([len(set(df.iloc[row]["row_uid"].tolist())) for row in I_diag])
print("distinct devices among the top-5 CHUNK hits, per query (5 = no redundancy):")
print(unique_device_counts.value_counts().sort_index())
print(f"\nmean distinct devices in top-5: {unique_device_counts.mean():.2f} / 5")


distinct devices among the top-5 CHUNK hits, per query (5 = no redundancy):
4     2
5    28
Name: count, dtype: int64

mean distinct devices in top-5: 4.93 / 5


In [8]:
def compute_hit_rate(index, eval_df, k=5):
    queries = eval_df["query"].tolist()
    query_vecs = model.encode(queries, batch_size=32, show_progress_bar=False).astype("float32")
    _, I = index.search(query_vecs, k)

    hits = []
    for i, row in eval_df.iterrows():
        relevant_uids = set(row["relevant_row_uids"])
        retrieved_uids = set(df.iloc[I[i]]["row_uid"].astype(str).tolist())
        hits.append(len(retrieved_uids & relevant_uids) > 0)
    return sum(hits) / len(hits)

hit_rate_results = []
for idx, name in [(index_flat, "IndexFlatL2"), (index_ivf, "IndexIVFFlat"), (index_hnsw, "IndexHNSWFlat"),
                   (index_pq, "IndexPQ"), (index_ivfpq, "IndexIVFPQ")]:
    hr = compute_hit_rate(idx, eval_df_correct, k=5)
    hit_rate_results.append({"index": name, "hit_rate": hr})
    print(f"{name:20s} hit rate: {hr:.3f}")

hit_rate_df = pd.DataFrame(hit_rate_results)
hit_rate_df.to_csv("m3_hit_rate_5way.csv", index=False)

final_table = results_df.merge(hit_rate_df, on="index")
final_table.to_csv("m3_final_5way_comparison.csv", index=False)
print()
print(final_table.to_string(index=False))


IndexFlatL2          hit rate: 0.533
IndexIVFFlat         hit rate: 0.500
IndexHNSWFlat        hit rate: 0.600
IndexPQ              hit rate: 0.433
IndexIVFPQ           hit rate: 0.333

        index  recall_at_5  latency_ms  build_time_s  memory_mb  hit_rate
  IndexFlatL2     1.000000    0.363015      0.005100   6.109906  0.533333
 IndexIVFFlat     0.826667    0.095990      0.063356   6.215442  0.500000
IndexHNSWFlat     0.846667    0.115586      0.349854   7.189672  0.600000
      IndexPQ     0.133333    0.123185      0.211869   0.117699  0.433333
   IndexIVFPQ     0.220000    0.060093      0.306935   0.243849  0.333333


In [9]:
def precision_recall_at_k(index, eval_df, k=5):
    queries = eval_df["query"].tolist()
    query_vecs = model.encode(queries, batch_size=32, show_progress_bar=False).astype("float32")
    _, I = index.search(query_vecs, k)

    precisions, recalls = [], []
    for i, row in eval_df.iterrows():
        retrieved_uids = set(df.iloc[I[i]]["row_uid"].astype(str).tolist())
        relevant_uids = set(row["relevant_row_uids"])
        if len(relevant_uids) == 0:
            continue
        hits = len(retrieved_uids & relevant_uids)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant_uids))

    return np.mean(precisions), np.mean(recalls)


pr_results = []
for idx, name in [(index_flat, "IndexFlatL2"), (index_ivf, "IndexIVFFlat"), (index_hnsw, "IndexHNSWFlat"),
                   (index_pq, "IndexPQ"), (index_ivfpq, "IndexIVFPQ")]:
    p, r = precision_recall_at_k(idx, eval_df_correct, k=5)
    pr_results.append({"index": name, "precision_at_5": p, "recall_at_5": r})
    print(f"{name:20s} | precision@5: {p:.3f} | recall@5: {r:.3f}")

pr_df = pd.DataFrame(pr_results)
pr_df.to_csv("m3_precision_recall_5way.csv", index=False)

table_6_2_full = pr_df.merge(hit_rate_df, on="index")
table_6_2_full.to_csv("table_6_2_full_5way.csv", index=False)
print()
print(table_6_2_full.to_string(index=False))


IndexFlatL2          | precision@5: 0.207 | recall@5: 0.025
IndexIVFFlat         | precision@5: 0.207 | recall@5: 0.025
IndexHNSWFlat        | precision@5: 0.253 | recall@5: 0.028
IndexPQ              | precision@5: 0.213 | recall@5: 0.019
IndexIVFPQ           | precision@5: 0.153 | recall@5: 0.021

        index  precision_at_5  recall_at_5  hit_rate
  IndexFlatL2        0.206667     0.024870  0.533333
 IndexIVFFlat        0.206667     0.024609  0.500000
IndexHNSWFlat        0.253333     0.027612  0.600000
      IndexPQ        0.213333     0.018694  0.433333
   IndexIVFPQ        0.153333     0.021393  0.333333


In [10]:
faiss.write_index(index_hnsw, "laptop_index_hnsw.faiss")
df[["chunk_id", "row_uid", "title", "chunk_text"]].to_csv("faiss_id_lookup.csv", index=False)
eval_df_correct.to_csv("eval_queries_final_regenerated.csv", index=False)

print("Saved: m3_index_comparison_5way.csv, m3_hit_rate_5way.csv, m3_final_5way_comparison.csv,")
print("       m3_precision_recall_5way.csv, table_6_2_full_5way.csv,")
print("       laptop_index_hnsw.faiss, faiss_id_lookup.csv, eval_queries_final_regenerated.csv")
print("\n(eval_queries_final_regenerated.csv's relevant_row_uids column reads back with")
print(" ast.literal_eval, not str.split — it's a stringified Python list, not a delimited string.)")

try:
    from google.colab import files
    for f in ["m3_final_5way_comparison.csv", "table_6_2_full_5way.csv",
              "eval_queries_final_regenerated.csv", "laptop_index_hnsw.faiss", "faiss_id_lookup.csv"]:
        files.download(f)
except ImportError:
    print("(Not in Colab — files are saved in the working directory; download manually if on Kaggle.)")


Saved: m3_index_comparison_5way.csv, m3_hit_rate_5way.csv, m3_final_5way_comparison.csv,
       m3_precision_recall_5way.csv, table_6_2_full_5way.csv,
       laptop_index_hnsw.faiss, faiss_id_lookup.csv, eval_queries_final_regenerated.csv

(eval_queries_final_regenerated.csv's relevant_row_uids column reads back with
 ast.literal_eval, not str.split — it's a stringified Python list, not a delimited string.)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>